# Kidney CT Classification — PyTorch (Google Colab)
**Cyst vs Normal vs Stone vs Tumor**

Research prototype only — not a diagnostic tool. See limitations at the end.

**How to use this notebook in Colab:**
1. Runtime → Change runtime type → GPU (T4 is fine).
2. Run cells top to bottom.
3. Upload your `Split_Images` folder (train/val/test, each with Cyst/Normal/Stone/Tumor subfolders) to Google Drive, e.g. `MyDrive/ketney stone/Split_Images`.
4. Update `DATA_ROOT` in the config cell below to match your Drive path.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Install/verify dependencies (Colab usually has these already)
!pip install -q torchmetrics grad-cam


In [ ]:
# 3. Imports
import os, copy, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score, MulticlassAUROC, MulticlassConfusionMatrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# 4. Config — EDIT THIS PATH to match your Google Drive folder
DATA_ROOT = "/content/drive/MyDrive/ketney stone/Split_Images"  # contains train/val/test
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]  # will be overwritten by ImageFolder's sorted order below
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS_HEAD = 10      # phase 1: frozen backbone
NUM_EPOCHS_FINETUNE = 10  # phase 2: unfrozen last layers
LR_HEAD = 1e-3
LR_FINETUNE = 1e-5
NUM_WORKERS = 2


## 5. Sanity check: class counts per split
Run this first. Verify counts look reasonable and roughly balanced (or note the imbalance so we can weight the loss).

In [ ]:
for split_name, split_dir in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    print(f"--- {split_name} ---")
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
            print(f"  {cls}: {n}")


## 6. Transforms and Datasets
- Resize to 224x224 (ResNet50 default input size)
- Normalize with ImageNet stats (required for pretrained weights)
- Augmentation on TRAIN only: mild rotation, zoom (via RandomResizedCrop), contrast jitter
- No augmentation on val/test — only resize + normalize
- **No horizontal flip by default** — many kidney CT slices are not left/right symmetric in a way that's safe to flip for classification purposes with mixed axial/coronal views. Enable it only if you've confirmed it's anatomically appropriate for your dataset.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # CT slices are grayscale; replicate to 3ch for pretrained nets
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

CLASS_NAMES = train_dataset.classes  # authoritative order, alphabetical: Cyst, Normal, Stone, Tumor
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


## 7. Class weights (for imbalance)
Computed from the train split only. Passed into the loss function so minority classes aren't ignored.

In [ ]:
from collections import Counter
targets = [label for _, label in train_dataset.samples]
counts = Counter(targets)
counts_sorted = [counts[i] for i in range(NUM_CLASSES)]
print("Train counts per class:", dict(zip(CLASS_NAMES, counts_sorted)))

total = sum(counts_sorted)
class_weights = torch.tensor([total / (NUM_CLASSES * c) for c in counts_sorted], dtype=torch.float32).to(device)
print("Class weights:", class_weights)


## 8. Visualize a few samples
Always look at your data before training. Confirms transforms and labels look correct.

In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (img_tensor * std + mean).clamp(0,1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12,6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1,2,0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 9. Model — ResNet50 transfer learning
Two-phase training:
1. **Head-only**: freeze backbone, train a new classifier head.
2. **Fine-tune**: unfreeze the last residual block(s), train with a much smaller LR.

This is the recommended baseline. A comparison model (EfficientNet or a from-scratch CNN) is included further below as an optional extension.

In [ ]:
def build_model(num_classes, freeze_backbone=True):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes)
    )
    return model.to(device)

model = build_model(NUM_CLASSES, freeze_backbone=True)
print(model.fc)


## 10. Training / evaluation loop utilities

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, scheduler=None, patience=5):
    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
              f"{time.time()-t0:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history


## 11. Phase 1 — Train the classifier head (backbone frozen)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

model, history_head = train_model(model, train_loader, val_loader, criterion, optimizer,
                                   num_epochs=NUM_EPOCHS_HEAD, scheduler=scheduler, patience=4)


## 12. Phase 2 — Fine-tune (unfreeze last block, small LR)

In [ ]:
# Unfreeze layer4 (last residual block) and the fc head
for name, param in model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        param.requires_grad = True

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.5, patience=2)

model, history_ft = train_model(model, train_loader, val_loader, criterion, optimizer_ft,
                                 num_epochs=NUM_EPOCHS_FINETUNE, scheduler=scheduler_ft, patience=4)


In [ ]:
# Save the trained model to Drive so it persists after the Colab session ends
SAVE_PATH = os.path.join(DATA_ROOT, "..", "resnet50_kidney_classifier.pt")
torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES}, SAVE_PATH)
print("Saved to", SAVE_PATH)


## 13. Plot training curves

In [ ]:
def plot_history(h1, h2, title_prefix=""):
    train_loss = h1["train_loss"] + h2["train_loss"]
    val_loss   = h1["val_loss"] + h2["val_loss"]
    train_acc  = h1["train_acc"] + h2["train_acc"]
    val_acc    = h1["val_acc"] + h2["val_acc"]

    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(train_loss, label="train"); axes[0].plot(val_loss, label="val")
    axes[0].axvline(len(h1["train_loss"])-0.5, color='gray', linestyle='--', label="fine-tune starts")
    axes[0].set_title(f"{title_prefix}Loss"); axes[0].legend()

    axes[1].plot(train_acc, label="train"); axes[1].plot(val_acc, label="val")
    axes[1].axvline(len(h1["train_acc"])-0.5, color='gray', linestyle='--')
    axes[1].set_title(f"{title_prefix}Accuracy"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_head, history_ft)


## 14. Evaluation on the held-out TEST set
Reports accuracy, per-class precision/recall/F1, macro ROC-AUC, and the confusion matrix. Accuracy alone is not sufficient for a 4-class medical imaging task — always check per-class recall, since missing a Tumor case is far more costly than missing a Cyst.

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels)
        all_probs.append(probs.cpu())

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_probs = torch.cat(all_probs)

acc_metric = MulticlassAccuracy(num_classes=NUM_CLASSES, average='macro')
prec_metric = MulticlassPrecision(num_classes=NUM_CLASSES, average=None)
rec_metric = MulticlassRecall(num_classes=NUM_CLASSES, average=None)
f1_metric = MulticlassF1Score(num_classes=NUM_CLASSES, average=None)
auroc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average='macro')
cm_metric = MulticlassConfusionMatrix(num_classes=NUM_CLASSES)

print(f"Overall (macro) accuracy: {acc_metric(all_preds, all_labels):.4f}")
print(f"Macro ROC-AUC: {auroc_metric(all_probs, all_labels):.4f}\n")

prec = prec_metric(all_preds, all_labels)
rec = rec_metric(all_preds, all_labels)
f1 = f1_metric(all_preds, all_labels)

print(f"{'Class':<10}{'Precision':<12}{'Recall':<12}{'F1':<12}")
for i, cls in enumerate(CLASS_NAMES):
    print(f"{cls:<10}{prec[i]:<12.4f}{rec[i]:<12.4f}{f1[i]:<12.4f}")


In [ ]:
# Confusion matrix plot
cm = cm_metric(all_preds, all_labels).numpy()

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, int(cm[i,j]), ha="center", va="center",
                 color="white" if cm[i,j] > cm.max()/2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()


## 15. Grad-CAM — Explainability
Visualizes which regions of the CT image the model used to make its prediction. **Important sanity check**: confirm the model is attending to the kidney region, not scanner borders, text overlays, or other artifacts (a common failure mode called "shortcut learning" in medical imaging datasets).

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

def show_gradcam(dataset, index):
    img_tensor, label = dataset[index]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    grayscale_cam = cam(input_tensor=input_tensor)[0]
    rgb_img = denormalize(img_tensor).permute(1,2,0).numpy()
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    model.eval()
    with torch.no_grad():
        pred = model(input_tensor).argmax(dim=1).item()

    fig, axes = plt.subplots(1, 2, figsize=(8,4))
    axes[0].imshow(rgb_img); axes[0].set_title(f"True: {CLASS_NAMES[label]}"); axes[0].axis("off")
    axes[1].imshow(visualization); axes[1].set_title(f"Pred: {CLASS_NAMES[pred]} (Grad-CAM)"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Show a few random test examples
for idx in random.sample(range(len(test_dataset)), 4):
    show_gradcam(test_dataset, idx)


## 16. (Optional) Comparison model — EfficientNet-B0
Useful for a comparison table in your report. Reuses the same training loop.

In [ ]:
def build_efficientnet(num_classes, freeze_backbone=True):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in m.parameters():
            p.requires_grad = False
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes)
    )
    return m.to(device)

# Uncomment to train:
# eff_model = build_efficientnet(NUM_CLASSES, freeze_backbone=True)
# optimizer_eff = optim.Adam(filter(lambda p: p.requires_grad, eff_model.parameters()), lr=LR_HEAD)
# eff_model, eff_history = train_model(eff_model, train_loader, val_loader, criterion, optimizer_eff, num_epochs=NUM_EPOCHS_HEAD)


## 17. Limitations & Ethical Notes (include in your report)
- **This is a research prototype, not a diagnostic tool.** It has not been clinically validated.
- **Possible data leakage**: if these CT slices were extracted per-patient, sequential slices from the same patient may appear in both train and test, inflating reported accuracy. Verify patient IDs if metadata is available.
- **No external validation**: results reflect one dataset's distribution; performance is not guaranteed to generalize to scans from different hospitals, scanners, or populations.
- **Grad-CAM should be checked, not just shown**: if attention maps consistently fall outside the kidney, treat metrics with suspicion regardless of how high they are.
- **Mixed axial/coronal views** in the same class folders may make the task easier or harder than a single-view dataset — mention this explicitly as a dataset characteristic.
- Report all metrics with confidence intervals or multiple-seed runs if possible, rather than a single accuracy number.
